# BERT NER Model — Transcript Entity Extraction

This notebook trains a BERT-based token classification model to extract
structured information (course codes, names, grades, semesters) from
academic transcripts using BIO-tagged NER.

**Runtime:** GPU (T4 or better) — Go to *Runtime → Change runtime type → GPU*

## 1. Setup

In [ ]:
!pip install -q transformers seqeval tabulate matplotlib

In [ ]:
# Clone the repository (change URL to your repo)
!git clone https://github.com/your-username/APS360-TranscriptClassifier.git
%cd APS360-TranscriptClassifier

In [ ]:
import sys, os, json
import torch
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# Verify GPU
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Data Exploration

In [ ]:
from evaluation.data_loader import load_dataset, group_by_template

all_samples = load_dataset("transcript_generator/output")
groups = group_by_template(all_samples)

print(f"Total samples: {len(all_samples)}")
print(f"Total templates: {len(groups)}")
print(f"Samples per template: {[len(v) for v in groups.values()][:5]}... (showing first 5)")

In [ ]:
# Token length distribution
lengths = [len(s["tokens"]) for s in all_samples]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.hist(lengths, bins=40, edgecolor="black", alpha=0.7)
ax1.axvline(512, color="red", linestyle="--", label=f"BERT limit (512)")
ax1.set_xlabel("Token count")
ax1.set_ylabel("Frequency")
ax1.set_title("Token Length Distribution")
ax1.legend()

# Label distribution
tag_counts = Counter()
for s in all_samples:
    tag_counts.update(s["ner_tags"])

tags = sorted(tag_counts.keys())
counts = [tag_counts[t] for t in tags]
colors = ["#999" if t == "O" else "#4CAF50" for t in tags]
ax2.barh(tags, counts, color=colors, edgecolor="black")
ax2.set_xlabel("Count")
ax2.set_title("BIO Label Distribution")

fig.tight_layout()
plt.show()

print(f"\nMin: {min(lengths)}, Max: {max(lengths)}, Mean: {np.mean(lengths):.0f}, Median: {np.median(lengths):.0f}")
print(f"Samples > 512 tokens: {sum(1 for l in lengths if l > 512)} ({sum(1 for l in lengths if l > 512)/len(lengths)*100:.1f}%)")

## 3. Template-Level Data Split

In [ ]:
from model.data_split import split_by_template, save_split_info

train_samples, val_samples, test_samples, split_info = split_by_template(
    data_dir="transcript_generator/output",
)
save_split_info(split_info)

print(f"Train: {split_info['train_samples']} samples ({len(split_info['train_templates'])} templates)")
print(f"Val:   {split_info['val_samples']} samples ({len(split_info['val_templates'])} templates)")
print(f"Test:  {split_info['test_samples']} samples ({len(split_info['test_templates'])} templates)")
print(f"\nTrain templates: {split_info['train_templates'][:5]}...")
print(f"Val templates:   {split_info['val_templates']}")
print(f"Test templates:  {split_info['test_templates']}")

## 4. Create Datasets

In [ ]:
from transformers import BertTokenizerFast
from model.dataset import TranscriptNERDataset
from model.config import MODEL_NAME, MAX_SEQ_LEN, STRIDE, ID_TO_LABEL

tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)

train_ds = TranscriptNERDataset(train_samples, tokenizer, MAX_SEQ_LEN, STRIDE)
val_ds = TranscriptNERDataset(val_samples, tokenizer, MAX_SEQ_LEN, STRIDE)
test_ds = TranscriptNERDataset(test_samples, tokenizer, MAX_SEQ_LEN, STRIDE)

print(f"Train windows: {len(train_ds)}")
print(f"Val windows:   {len(val_ds)}")
print(f"Test windows:  {len(test_ds)}")

In [ ]:
# Sanity check: inspect one sample
sample = train_ds[0]
ids = sample["input_ids"]
labels = sample["labels"]

print("First 30 tokens and labels:")
for i in range(30):
    tok = tokenizer.convert_ids_to_tokens(ids[i].item())
    lab = ID_TO_LABEL[labels[i].item()] if labels[i].item() != -100 else "<ign>"
    print(f"  {tok:20s} → {lab}")

## 5. Train the Model

In [ ]:
from model.train import train

history = train(
    data_dir="transcript_generator/output",
    output_dir="model/checkpoints",
    num_epochs=10,
    batch_size=16,
    learning_rate=5e-5,
    use_fp16=True,
)

## 6. Learning Curves

In [ ]:
from model.utils import plot_learning_curves

plot_learning_curves(history, save_path="model/logs/learning_curves.png")

# Also display inline
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
epochs = range(1, len(history["train_loss"]) + 1)

ax1.plot(epochs, history["train_loss"], "o-", label="Train Loss")
ax1.plot(epochs, history["val_loss"], "s-", label="Val Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Training & Validation Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, history["val_entity_f1"], "o-", label="Val Entity F1", color="green")
ax2.plot(epochs, history["val_token_acc"], "s-", label="Val Token Acc", color="royalblue")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Score")
ax2.set_title("Validation Metrics")
ax2.set_ylim(0, 1.05)
ax2.legend()
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 7. Evaluate on Test Set

In [ ]:
from model.predict import BertNERPredictor
from model.config import BEST_CHECKPOINT_DIR
from seqeval.metrics import classification_report, f1_score as seqeval_f1

predictor = BertNERPredictor(checkpoint_dir="model/checkpoints/best")

# Run predictions on test set
all_true, all_pred = [], []
for sample in test_samples:
    tokens = sample["tokens"]
    true_tags = sample["ner_tags"]
    pred_tags = predictor.predict(tokens)
    all_true.append(true_tags)
    all_pred.append(pred_tags)

# Overall metrics
test_f1 = seqeval_f1(all_true, all_pred)
correct = sum(t == p for ts, ps in zip(all_true, all_pred) for t, p in zip(ts, ps))
total = sum(len(ts) for ts in all_true)
test_acc = correct / total

print(f"Test Token Accuracy: {test_acc:.4f}")
print(f"Test Entity F1:      {test_f1:.4f}")
print()
print(classification_report(all_true, all_pred, digits=4))

## 8. Qualitative Examples

In [ ]:
from evaluation.reconstructor import reconstruct_semesters
from tabulate import tabulate

def show_extraction(sample, pred_tags, title=""):
    """Display extracted courses from a transcript."""
    tokens = sample["tokens"]
    true_tags = sample["ner_tags"]
    tid = sample.get("metadata", {}).get("transcript_id", "unknown")

    true_sems = reconstruct_semesters(tokens, true_tags)
    pred_sems = reconstruct_semesters(tokens, pred_tags)

    print(f"\n{'='*60}")
    print(f"{title or tid}")
    print(f"{'='*60}")

    for label, sems in [("Ground Truth", true_sems), ("Predicted", pred_sems)]:
        print(f"\n--- {label} ---")
        for sem in sems:
            print(f"\n  {sem['semester_name']}")
            rows = [[c["code"], c["name"], c["grade"]] for c in sem["courses"]]
            if rows:
                print(tabulate(rows, headers=["Code", "Name", "Grade"], tablefmt="simple", stralign="left"))

# Show a few test examples
for i in range(min(3, len(test_samples))):
    show_extraction(
        test_samples[i],
        all_pred[i],
        title=f"Test Sample {i+1}",
    )

In [ ]:
# Show worst-performing test transcript
per_f1 = [
    seqeval_f1([t], [p])
    for t, p in zip(all_true, all_pred)
]
worst_idx = int(np.argmin(per_f1))
print(f"Worst test F1: {per_f1[worst_idx]:.4f}")
show_extraction(
    test_samples[worst_idx],
    all_pred[worst_idx],
    title=f"Worst Test Transcript (F1={per_f1[worst_idx]:.4f})",
)

## 9. Compare with Baseline

In [ ]:
sys.path.insert(0, "baseline_model")
from classifier import predict as baseline_predict

baseline_true, baseline_pred = [], []
for sample in test_samples:
    tokens = sample["tokens"]
    true_tags = sample["ner_tags"]
    pred_tags = baseline_predict(tokens)
    baseline_true.append(true_tags)
    baseline_pred.append(pred_tags)

baseline_f1 = seqeval_f1(baseline_true, baseline_pred)
b_correct = sum(t == p for ts, ps in zip(baseline_true, baseline_pred) for t, p in zip(ts, ps))
b_total = sum(len(ts) for ts in baseline_true)
baseline_acc = b_correct / b_total

print(f"{'Model':<20} {'Token Acc':>10} {'Entity F1':>10}")
print(f"{'-'*42}")
print(f"{'Rule-Based Baseline':<20} {baseline_acc:>10.4f} {baseline_f1:>10.4f}")
print(f"{'BERT NER':<20} {test_acc:>10.4f} {test_f1:>10.4f}")

## 10. Save Checkpoint

Save the trained model to Google Drive or download it.

In [ ]:
# Option A: Save to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r model/checkpoints/best /content/drive/MyDrive/aps360_bert_checkpoint

# Option B: Download as zip
!zip -r bert_checkpoint.zip model/checkpoints/best model/logs
# from google.colab import files
# files.download('bert_checkpoint.zip')
print("Checkpoint zipped. Uncomment files.download() to download.")